# Silver Layer

Esta capa toma los datos crudos de Bronze y los transforma en datos limpios,
tipificados y deduplicados, listos para alimentar la capa Gold.

Reglas aplicadas a todas las tablas:

- Deduplicación por clave primaria (se conserva la versión más reciente).
- Casteo a tipos correctos (DATE, TIMESTAMP, DECIMAL, BIGINT, BOOLEAN).
- TRIM de columnas de texto.
- Filtros de validez por tabla (montos > 0, fechas coherentes, rangos válidos).
- Eliminación de registros con PK nula.

Se usa `CREATE OR REPLACE TABLE` para que el notebook sea idempotente:
se puede correr múltiples veces sin errores.

## silver_users

Limpieza de la tabla de usuarios.

- Dedupe por `user_id` conservando el registro más reciente.
- Trim de email, name, country y company_name.
- Cast de `is_business` a BOOLEAN y `created_at` a TIMESTAMP.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_users AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY user_id
      ORDER BY created_at DESC
    ) AS rn
  FROM bronze.bronze_users
  WHERE user_id IS NOT NULL
)
SELECT
  CAST(user_id AS BIGINT)            AS user_id,
  TRIM(email)                        AS email,
  TRIM(name)                         AS name,
  TRIM(country)                      AS country,
  TRIM(user_type)                    AS user_type,
  CAST(is_business AS BOOLEAN)       AS is_business,
  TRIM(company_name)                 AS company_name,
  CAST(created_at AS TIMESTAMP)      AS created_at
FROM dedup
WHERE rn = 1;

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_users
FROM silver.silver_users;

## silver_destinations

Limpieza de la tabla de destinos turísticos.

- Dedupe por `destination_id`.
- Trim de campos descriptivos.
- La columna `description` (texto markdown extenso) se conserva en Silver pero se descartará en Gold.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_destinations AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY destination_id
      ORDER BY destination_id
    ) AS rn
  FROM bronze.bronze_destinations
  WHERE destination_id IS NOT NULL
)
SELECT
  CAST(destination_id AS BIGINT)        AS destination_id,
  TRIM(destination)                     AS destination,
  TRIM(country)                         AS country,
  TRIM(state_or_province)               AS state_or_province,
  TRIM(state_or_province_code)          AS state_or_province_code,
  description
FROM dedup
WHERE rn = 1;

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_destinations
FROM silver.silver_destinations;

## silver_properties

Limpieza de la tabla de propiedades.

- Dedupe por `property_id`.
- Cast de latitud/longitud a DOUBLE.
- Cast de `base_price` a DECIMAL(10,2).
- Filtros: `base_price > 0` y `max_guests > 0` para descartar registros inválidos.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_properties AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY property_id
      ORDER BY created_at DESC
    ) AS rn
  FROM bronze.bronze_properties
  WHERE property_id IS NOT NULL
)
SELECT
  CAST(property_id AS BIGINT)              AS property_id,
  CAST(host_id AS BIGINT)                  AS host_id,
  CAST(destination_id AS BIGINT)           AS destination_id,
  TRIM(title)                              AS title,
  TRIM(property_type)                      AS property_type,
  CAST(max_guests AS INT)                  AS max_guests,
  CAST(bedrooms AS INT)                    AS bedrooms,
  CAST(bathrooms AS INT)                   AS bathrooms,
  CAST(base_price AS DECIMAL(10,2))        AS base_price,
  CAST(property_latitude AS DOUBLE)        AS property_latitude,
  CAST(property_longitude AS DOUBLE)       AS property_longitude,
  CAST(created_at AS DATE)                 AS created_at
FROM dedup
WHERE rn = 1
  AND base_price > 0
  AND max_guests > 0;

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_properties
FROM silver.silver_properties;

## silver_bookings

Limpieza de la tabla principal de reservas.

- Dedupe por `booking_id` conservando el registro más reciente según `updated_at`.
- Cast de fechas a DATE y timestamps a TIMESTAMP.
- Filtros: `total_amount > 0` y `check_out > check_in`.
- Se agrega columna derivada `total_nights` para usarla luego en la fact table.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_bookings AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY booking_id
      ORDER BY updated_at DESC
    ) AS rn
  FROM bronze.bronze_bookings
  WHERE booking_id IS NOT NULL
)
SELECT
  CAST(booking_id AS BIGINT)             AS booking_id,
  CAST(user_id AS BIGINT)                AS user_id,
  CAST(property_id AS BIGINT)            AS property_id,
  CAST(check_in AS DATE)                 AS check_in,
  CAST(check_out AS DATE)                AS check_out,
  CAST(guests_count AS INT)              AS guests_count,
  CAST(total_amount AS DECIMAL(10,2))    AS total_amount,
  TRIM(status)                           AS status,
  CAST(created_at AS TIMESTAMP)          AS created_at,
  CAST(updated_at AS TIMESTAMP)          AS updated_at,
  DATEDIFF(
    CAST(check_out AS DATE),
    CAST(check_in AS DATE)
  )                                      AS total_nights
FROM dedup
WHERE rn = 1
  AND CAST(total_amount AS DECIMAL(10,2)) > 0
  AND CAST(check_out AS DATE) > CAST(check_in AS DATE);

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_bookings
FROM silver.silver_bookings;

## silver_payments

Limpieza de la tabla de pagos.

- Dedupe por `payment_id`.
- Cast de `amount` a DECIMAL(10,2).
- Cast de `payment_date` a TIMESTAMP.
- Filtro: `amount > 0` para eliminar pagos inválidos.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_payments AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY payment_id
      ORDER BY payment_date DESC
    ) AS rn
  FROM bronze.bronze_payments
  WHERE payment_id IS NOT NULL
)
SELECT
  CAST(payment_id AS BIGINT)           AS payment_id,
  CAST(booking_id AS BIGINT)           AS booking_id,
  CAST(amount AS DECIMAL(10,2))        AS amount,
  TRIM(payment_method)                 AS payment_method,
  TRIM(status)                         AS status,
  CAST(payment_date AS TIMESTAMP)      AS payment_date
FROM dedup
WHERE rn = 1
  AND CAST(amount AS DECIMAL(10,2)) > 0;

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_payments
FROM silver.silver_payments;

## silver_reviews

Limpieza de la tabla de reseñas.

- Dedupe por `review_id` conservando la versión más reciente.
- Trim del campo `comment`.
- Cast de `rating` a DECIMAL(3,1).
- Filtros: `is_deleted = false` y `rating BETWEEN 0 AND 5`.

In [ ]:
%sql
CREATE OR REPLACE TABLE silver.silver_reviews AS
WITH dedup AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY review_id
      ORDER BY updated_at DESC
    ) AS rn
  FROM bronze.bronze_reviews
  WHERE review_id IS NOT NULL
)
SELECT
  CAST(review_id AS BIGINT)         AS review_id,
  CAST(booking_id AS BIGINT)        AS booking_id,
  CAST(user_id AS BIGINT)           AS user_id,
  CAST(property_id AS BIGINT)       AS property_id,
  CAST(rating AS DECIMAL(3,1))      AS rating,
  TRIM(comment)                     AS comment,
  CAST(is_deleted AS BOOLEAN)       AS is_deleted,
  CAST(created_at AS TIMESTAMP)     AS created_at,
  CAST(updated_at AS TIMESTAMP)     AS updated_at
FROM dedup
WHERE rn = 1
  AND CAST(is_deleted AS BOOLEAN) = false
  AND CAST(rating AS DECIMAL(3,1)) BETWEEN 0 AND 5;

In [ ]:
%sql
SELECT COUNT(*) AS total_silver_reviews
FROM silver.silver_reviews;

## Validación final de Silver

Resumen del volumen de cada tabla Silver para verificar que todas se construyeron correctamente.

In [ ]:
%sql
SELECT 'silver_users'        AS tabla, COUNT(*) AS registros FROM silver.silver_users
UNION ALL
SELECT 'silver_destinations' AS tabla, COUNT(*) AS registros FROM silver.silver_destinations
UNION ALL
SELECT 'silver_properties'   AS tabla, COUNT(*) AS registros FROM silver.silver_properties
UNION ALL
SELECT 'silver_bookings'     AS tabla, COUNT(*) AS registros FROM silver.silver_bookings
UNION ALL
SELECT 'silver_payments'     AS tabla, COUNT(*) AS registros FROM silver.silver_payments
UNION ALL
SELECT 'silver_reviews'      AS tabla, COUNT(*) AS registros FROM silver.silver_reviews
ORDER BY tabla;

## Conclusión Silver

La capa Silver dejó las seis entidades limpias, deduplicadas y tipificadas:

- `silver.silver_users`
- `silver.silver_destinations`
- `silver.silver_properties`
- `silver.silver_bookings` (con `total_nights` derivado)
- `silver.silver_payments`
- `silver.silver_reviews`

Estas tablas son el insumo directo para la capa Gold, donde se construirá
el Star Schema con las dimensiones y la tabla de hechos `fact_reservas`.